#1. Read Bronze

In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

payments_df = spark.table(
    f"{catalog}.ecommerce_bronze.payments"
)

In [0]:
payments_df.printSchema()

#2. Clean + standardize

The payments dataset has:

- order_id
- payment_sequential
- payment_type
- payment_installments
- payment_value

We'll clean the strings and enforce the numeric types.

In [0]:
silver_payments = (
    payments_df
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("payment_type", F.lower(F.trim(F.col("payment_type"))))
)

#3. Write to ADLS

In [0]:
silver_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/payments"

silver_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

#4. Register in Unity Catalog

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_silver.payments
USING DELTA
LOCATION '{silver_path}'
""")